#### Importing required libraries 

In [1]:
from utils import Load_Rumours_Dataset_filtering_since_first_post
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import train_test_split,StratifiedKFold
from sklearn.metrics import *
import pandas as pd
import time
import optuna
from lightgbm import LGBMClassifier
import warnings
warnings.filterwarnings("ignore")
event_name ="sydneysiege"

In [2]:
file_path_replies = f"replies_{event_name}.pkl"
file_path_posts = f"posts_{event_name}.pkl"


#### Testing a single load 

In [19]:
processor = Load_Rumours_Dataset_filtering_since_first_post(file_path_replies, file_path_posts, time_cut=10000000)
processor.load_data()
processor.process_data()
train,test= processor.get_final_dataframes()


In [20]:
test.shape

(258, 10)

In [4]:
previous_node_count = 0

In [5]:
X_train  = train.drop(columns=['rumour'])
X_train = np.hstack([X_train.drop(columns=['embeddings_avg']).values, np.array(pd.DataFrame(X_train.embeddings_avg.tolist()))])
#X = np.hstack([X.drop(columns=['embeddings_avg']).values, np.array(pd.DataFrame(X.embeddings_avg.tolist()))])
y_train =train['rumour']

X_test  = test.drop(columns=['rumour'])
X_test_new = test.iloc[previous_node_count:].drop(columns=['rumour'])

X_test_new =  np.hstack([X_test_new.drop(columns=['embeddings_avg']).values, np.array(pd.DataFrame(X_test_new.embeddings_avg.tolist()))])
X_test = np.hstack([X_test.drop(columns=['embeddings_avg']).values, np.array(pd.DataFrame(X_test.embeddings_avg.tolist()))])


y_test =test['rumour']
y_test_new = test.iloc[previous_node_count:]['rumour']

previous_node_count = test.shape[0]
print(f"New Instances: {X_test_new.shape[0]}")

New Instances: 601


#### Tunning Light GBM

In [14]:
previous_node_count = 0

In [15]:
### Tuning with all data
processor = Load_Rumours_Dataset_filtering_since_first_post(file_path_replies, file_path_posts, time_cut=3*24*60)
processor.load_data()
processor.process_data()
train,test= processor.get_final_dataframes()


In [16]:
X_train  = train.drop(columns=['rumour'])
X_train = np.hstack([X_train.drop(columns=['embeddings_avg']).values, np.array(pd.DataFrame(X_train.embeddings_avg.tolist()))])
#X = np.hstack([X.drop(columns=['embeddings_avg']).values, np.array(pd.DataFrame(X.embeddings_avg.tolist()))])
y_train =train['rumour']

X_test  = test.drop(columns=['rumour'])
X_test_new = test.iloc[previous_node_count:].drop(columns=['rumour'])

X_test_new =  np.hstack([X_test_new.drop(columns=['embeddings_avg']).values, np.array(pd.DataFrame(X_test_new.embeddings_avg.tolist()))])
X_test = np.hstack([X_test.drop(columns=['embeddings_avg']).values, np.array(pd.DataFrame(X_test.embeddings_avg.tolist()))])


y_test =test['rumour']
y_test_new = test.iloc[previous_node_count:]['rumour']

previous_node_count = test.shape[0]
print(f"New Instances: {X_test_new.shape[0]}")

New Instances: 303


In [ ]:

n_train = len(X_train)

def objective(trial, X, y):
    param_grid = {
    # number of trees: keep reasonable upper bound for small dataset
    "n_estimators": trial.suggest_int("n_estimators", 50, 300, step=25),

    # learning rate: similar but slightly wider
    "learning_rate": trial.suggest_loguniform("learning_rate", 1e-4, 1e-2),

    # tree complexity: allow more variety but avoid huge trees for tiny data
    "num_leaves": trial.suggest_int("num_leaves", 3, 64, step=1),
    "max_depth": trial.suggest_int("max_depth", 2, 5),

    # min data in leaf: relative to training size (never larger than n_train)
    # lower bound 1, upper bound floor(n_train * 0.2) ensures splits are possible
    "min_data_in_leaf": trial.suggest_int(
        "min_data_in_leaf",
        1,
        max(2, int(max(2, n_train * 0.2))),
    ),

    # regularization: keep wide but avoid 0 lower bound
    "lambda_l1": trial.suggest_loguniform("lambda_l1", 1e-4, 100.0),
    "lambda_l2": trial.suggest_loguniform("lambda_l2", 1e-4, 100.0),

    # subsampling / feature fraction: use continuous suggestions
    "bagging_fraction": trial.suggest_uniform("bagging_fraction", 0.4, 1.0),
    # bagging frequency: allow 0 (no bagging) up to small integers
    "bagging_freq": trial.suggest_int("bagging_freq", 0, 10),
    "feature_fraction": trial.suggest_uniform("feature_fraction", 0.4, 1.0),
    }
    
    cv = StratifiedKFold(n_splits = 4, shuffle = True, random_state = 1337)
    cv_scores = np.empty(4)
    
    for idx, (train_idx, test_idx) in enumerate(cv.split(X, y)):
        X_train_fold, X_test_fold = X.iloc[train_idx], X.iloc[test_idx]
        y_train_fold, y_test_fold = y[train_idx], y[test_idx]

        model = LGBMClassifier(objective="binary", **param_grid,verbosity=-1,
                            seed= 1337,
                            feature_fraction_seed= 1337,
                            bagging_seed= 1337,
                            drop_seed= 1337,
                            data_random_seed= 1337
                            #class_weight= {0: neg_class_weight, 1: 1.0}
                              )
        #puning_callback = optuna.integration.LightGBMPruningCallback(trial, "auc")
        
        model.fit(
                X_train_fold,
                y_train_fold,
                eval_set = [(X_test_fold, y_test_fold)],
                eval_metric="auc")

        
        
        y_train_prob = model.predict_proba(X_train_fold)[:, 1]
        thresholds = np.linspace(0.01, 0.99, 100)
        f1_scores = [f1_score(y_train_fold, (y_train_prob > t).astype(int)) for t in thresholds]
        best_idx = np.argmax(f1_scores)
        best_threshold = thresholds[best_idx]


        y_train_pred = (y_train_prob > best_threshold).astype(int)
        f1_score_idx = f1_score(y_train_fold, y_train_pred)
       

        # Report and prune
        trial.report(f1_score_idx, step=idx)
        if trial.should_prune():
            raise optuna.TrialPruned()
        
        ### get F-beta score on top 

        cv_scores[idx] = f1_score_idx

    return np.mean(cv_scores)




In [ ]:
# Run study
start_time = time.time()

study = optuna.create_study(direction="maximize", study_name="LGBM Charlie Hebdo",
                            sampler=optuna.samplers.TPESampler(seed=111113857),
                            pruner=optuna.pruners.MedianPruner())
func = lambda trial: objective(trial, pd.DataFrame(X_train), y_train.astype(int))
study.optimize(func, n_trials=100)

end_time = time.time()

hyper_tuning_time = end_time - start_time

print(f"\tBest value (bcr1p_sum): {study.best_value:.5f}")
print(f"\tBest params:")

for key, value in study.best_params.items():
    print(f"\t\t{key}: {value}")


#### Example  training

In [3]:
best_params = {
    "objective": "binary",
    "metric": ["auc", "average_precision"],

    # Imbalance handling (~15% bad rate)
    "is_unbalance": True,
    # alternatively: "scale_pos_weight": 0.15 / 0.85 ≈ 0.176 (but don't use both)

    # Tree complexity (important with only ~10 features)
    "num_leaves": 15,
    "max_depth": 4,

    # Regularization (helps prevent overfitting, especially with embeddings)
    "min_data_in_leaf": 100,
    "lambda_l1": 1.0,
    "lambda_l2": 5.0,

    # Feature / row sampling
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,

    # Learning control
    "learning_rate": 0.03,
    "n_estimators": 1000,

    # Binning (important for embedding feature)
    "max_bin": 255,

    # Stability
    "verbosity": -1,
    "seed": 42
}

In [22]:

model = lgb.LGBMClassifier(
    objective="binary",
    boosting_type="gbdt",
   **best_params,
    n_jobs=-1,
    random_state=42,
    verbose=-1
)
# Train the model
model.fit(
    X_train,
    y_train,
    eval_metric=["binary_logloss", "auc"],
)


LGBMClassifier(bagging_fraction=0.7521688515598353, bagging_freq=1,
               feature_fraction=0.8860074803941028,
               lambda_l1=0.0002623314818549636, lambda_l2=1.2647354666343253,
               learning_rate=0.008750545843056286, max_depth=4,
               min_data_in_leaf=6, n_estimators=200, n_jobs=-1, num_leaves=30,
               objective='binary', random_state=42, verbose=-1)

In [23]:
y_train_prob = model.predict_proba(X_train)[:, 1]
y_test_prob = model.predict_proba(X_test)[:, 1]
y_test_new_prob = model.predict_proba(X_test_new)[:, 1]

thresholds = np.linspace(0.01, 0.99, 100)
f1_scores = [f1_score(y_train, (y_train_prob > t).astype(int)) for t in thresholds]
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]

y_train_pred = (y_train_prob > best_threshold).astype(int)
y_test_pred = (y_test_prob > best_threshold).astype(int)
y_test_new_pred = (y_test_new_prob > best_threshold).astype(int)

# Evaluation function
def evaluate(y_true, y_pred, y_prob, label=""):
    print(f"  - Accuracy:  {accuracy_score(y_true, y_pred):.4f}")
    print(f"  - Precision: {precision_score(y_true, y_pred):.4f}")
    print(f"  - Recall:    {recall_score(y_true, y_pred):.4f}")
    print(f"  - AUC:       {roc_auc_score(y_true, y_prob):.4f}")
    print("")

# Show metrics
print('Train Set: ')
evaluate(y_train, y_train_pred, y_train_prob, label="Train")
print('Test Set: ')
evaluate(y_test, y_test_pred, y_test_prob, label="Test")

Train Set: 
  - Accuracy:  0.9661
  - Precision: 0.9018
  - Recall:    0.8860
  - AUC:       0.9930

Test Set: 
  - Accuracy:  0.5776
  - Precision: 0.8947
  - Recall:    0.1189
  - AUC:       0.7885



#### Setting MLflow Experiment

In [3]:
from datetime import date
import mlflow

today = date.today()
formatted_today = today.strftime("%Y-%m-%d")


mlflow.set_experiment(f"Light GBM {formatted_today} {event_name} run")

2026/07/05 18:01:10 INFO mlflow.tracking.fluent: Experiment with name 'Light GBM 2026-07-05 sydneysiege run' does not exist. Creating a new experiment.


<Experiment: artifact_location='/workspaces/rumour-detection-gnn/mlruns/123', creation_time=1783274470976, experiment_id='123', last_update_time=1783274470976, lifecycle_stage='active', name='Light GBM 2026-07-05 sydneysiege run', tags={}, workspace='default'>

#### Loading dataset statistics to get the final time cut 

In [4]:
import pandas as pd
df_posts_by_time_cut = pd.read_pickle(f'replies_{event_name}.pkl')

df_posts_by_time_cut['min_since_fst_post'] = round(
            (df_posts_by_time_cut['time'] - df_posts_by_time_cut['time'].min()).dt.total_seconds() / 60, 2)


In [5]:
df_metrics = df_posts_by_time_cut[['id','time','rumour','min_since_fst_post']].drop_duplicates().sort_values(by='time')

In [6]:
file_path_replies = f"replies_{event_name}.pkl"
file_path_posts = f"posts_{event_name}.pkl"


processor = Load_Rumours_Dataset_filtering_since_first_post(file_path_replies, file_path_posts, time_cut=10000)
processor.load_data()
processor.process_data()
train,test= processor.get_final_dataframes()


In [7]:

start = test.min_since_fst_post.min()
end = test.min_since_fst_post.max()
duration = end-start
experiment_time= duration+60



In [8]:
start

np.float64(268.53)

In [10]:
end

np.float64(1016.1)

In [8]:
experiment_time

np.float64(368.72)

In [8]:
def compute_metrics_custom(df, prob_col='prob', target_col='rumour'):
    df = df.copy()
    df = df.sort_values(prob_col, ascending=False).reset_index(drop=True)

    total_frauds = df[target_col].sum()
    total_records = df.shape[0]
    n = len(df)

    # ✅ Bins from 1% to 100% in 1% steps
    bins = np.arange(0.01, 1.01, 0.01)

    results = []

    for p in bins:
        cutoff = int(np.ceil(n * p))
        subset = df.iloc[:cutoff]

        frauds = subset[target_col].sum()
        records = len(subset)

        results.append({
            'percentile': round(p * 100, 0),
            'records': records,
            'frauds_captured': frauds,
            'capture_rate': frauds / total_frauds if total_frauds > 0 else 0,
            'bad_rate': frauds / records if records > 0 else 0,
            'false_positive_rate': (records - frauds) / total_records if records > 0 else 0
        })

    df_out = (
        pd.DataFrame(results)
        .drop_duplicates(subset='percentile')
        .sort_values('percentile')
        .reset_index(drop=True)
    )

    return df_out

In [9]:
best_params = {
    "objective": "binary",
    "metric": ["auc", "average_precision"],

    # Imbalance handling (~15% bad rate)
    # alternatively: "scale_pos_weight": 0.15 / 0.85 ≈ 0.176 (but don't use both)

    # Tree complexity (important with only ~10 features)
    "num_leaves": 15,
    "max_depth": 4,

    # Regularization (helps prevent overfitting, especially with embeddings)
    "min_data_in_leaf": 100,
    "lambda_l1": 1.0,
    "lambda_l2": 5.0,

    # Feature / row sampling
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,

    # Learning control
    "learning_rate": 0.03,
    "n_estimators": 1000,

    # Binning (important for embedding feature)
    "max_bin": 255,

    # Stability
    "verbosity": -1,
    "seed": 42
}

In [11]:
import mlflow
import mlflow.lightgbm
import warnings
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score, f1_score
import lightgbm as lgb

previous_node_count = 0

for time_cut in np.linspace(0, int(experiment_time), 50):
    time_cut = int(time_cut)
    print(f"\n=== Time Cut: {time_cut} minutes ===")
    
    processor = Load_Rumours_Dataset_filtering_since_first_post(file_path_replies, file_path_posts, time_cut=time_cut)
    processor.load_data()
    processor.process_data()
    train, test = processor.get_final_dataframes()

    # Prepare features and labels
    X_train  = train.drop(columns=['rumour'])
    X_train = np.hstack([X_train.drop(columns=['embeddings_avg']).values, np.array(pd.DataFrame(X_train.embeddings_avg.tolist()))])
    #X = np.hstack([X.drop(columns=['embeddings_avg']).values, np.array(pd.DataFrame(X.embeddings_avg.tolist()))])
    y_train =train['rumour']
    
    X_test  = test.drop(columns=['rumour'])
    X_test_new = test.iloc[previous_node_count:].drop(columns=['rumour'])
    
    X_test_new =  np.hstack([X_test_new.drop(columns=['embeddings_avg']).values, np.array(pd.DataFrame(X_test_new.embeddings_avg.tolist()))])
    X_test = np.hstack([X_test.drop(columns=['embeddings_avg']).values, np.array(pd.DataFrame(X_test.embeddings_avg.tolist()))])
    
    
    y_test =test['rumour']
    y_test_new = test.iloc[previous_node_count:]['rumour']
    
    previous_node_count = test.shape[0]
    
    print(f"New Instances: {X_test_new.shape[0]}")


    model = lgb.LGBMClassifier(
    
           **best_params
      
    )

    with mlflow.start_run(run_name=f"time_cut_{time_cut}"):
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            model.fit(
                X_train, y_train,
                eval_metric=["binary_logloss", "auc"]
            )

        # Get predicted probabilities
        y_train_prob = model.predict_proba(X_train)[:, 1]
        y_test_prob = model.predict_proba(X_test)[:, 1]
        current_df_metrics = df_metrics.iloc[X_train.shape[0]:X_train.shape[0]+X_test.shape[0]]
        current_df_metrics['prob'] = y_test_prob
        df_metrics_by_bucket = compute_metrics_custom(current_df_metrics)
        
        if X_test_new.shape[0] >0:
            y_test_new_prob = model.predict_proba(X_test_new)[:, 1]

        # Find best threshold maximizing F1 score on training data
        thresholds = np.linspace(0.01, 0.99, 100)
        f1_scores = [f1_score(y_train, (y_train_prob > t).astype(int)) for t in thresholds]
        best_idx = np.argmax(f1_scores)
        best_threshold = thresholds[best_idx]

        # Apply optimal threshold
        y_train_pred = (y_train_prob > best_threshold).astype(int)
        y_test_pred = (y_test_prob > best_threshold).astype(int)
        y_test_new_pred = (y_test_new_prob > best_threshold).astype(int)

        # Log train metrics
        mlflow.log_metric("train_accuracy", accuracy_score(y_train, y_train_pred))
        mlflow.log_metric("train_precision", precision_score(y_train, y_train_pred))
        mlflow.log_metric("train_recall", recall_score(y_train, y_train_pred))
        mlflow.log_metric("train_f1", f1_score(y_train, y_train_pred))
        mlflow.log_metric("train_auc", roc_auc_score(y_train, y_train_prob))

        # Log test metrics
        mlflow.log_metric("final_acc", accuracy_score(y_test, y_test_pred))
        mlflow.log_metric("final_precision", precision_score(y_test, y_test_pred))
        mlflow.log_metric("final_recall", recall_score(y_test, y_test_pred))
        mlflow.log_metric("final_f1", f1_score(y_test, y_test_pred))
        mlflow.log_metric("final_auc", roc_auc_score(y_test, y_test_prob))
        mlflow.log_metric("new_posts", X_test_new.shape[0])
        df_metrics_by_bucket.to_csv(f"metrics_by_bucket_{event_name}_LGBM.csv", index=False)
        mlflow.log_artifact(f"metrics_by_bucket_{event_name}_LGBM.csv")

        if X_test_new.shape[0] >0:
            mlflow.log_metric("curr_precision", precision_score(y_test_new, y_test_new_pred))
            mlflow.log_metric("curr_recall", recall_score(y_test_new, y_test_new_pred))
            mlflow.log_metric("curr_acc", accuracy_score(y_test_new, y_test_new_pred))
        else:
            mlflow.log_metric("curr_precision", 0)
            mlflow.log_metric("curr_recall",0)
            mlflow.log_metric("curr_acc", 0)
            

        # Log threshold and time_cut
        mlflow.log_metric("optimal_threshold", best_threshold)
        mlflow.log_metric("time_cut", time_cut)



=== Time Cut: 0 minutes ===
New Instances: 1

=== Time Cut: 41 minutes ===
New Instances: 10

=== Time Cut: 83 minutes ===
New Instances: 4

=== Time Cut: 125 minutes ===
New Instances: 4

=== Time Cut: 167 minutes ===
New Instances: 3

=== Time Cut: 208 minutes ===
New Instances: 4

=== Time Cut: 250 minutes ===
New Instances: 1

=== Time Cut: 292 minutes ===
New Instances: 0

=== Time Cut: 334 minutes ===
New Instances: 4

=== Time Cut: 375 minutes ===
New Instances: 0

=== Time Cut: 417 minutes ===
New Instances: 0

=== Time Cut: 459 minutes ===
New Instances: 0

=== Time Cut: 501 minutes ===
New Instances: 0

=== Time Cut: 543 minutes ===
New Instances: 0

=== Time Cut: 584 minutes ===
New Instances: 0

=== Time Cut: 626 minutes ===
New Instances: 0

=== Time Cut: 668 minutes ===
New Instances: 0

=== Time Cut: 710 minutes ===
New Instances: 0

=== Time Cut: 751 minutes ===
New Instances: 0

=== Time Cut: 793 minutes ===
New Instances: 0

=== Time Cut: 835 minutes ===
New Instance

In [10]:
new_posts_times = np.sort(
    np.unique(
        np.ceil(
            df_metrics[
                (df_metrics.min_since_fst_post >= start ) &
                (df_metrics.min_since_fst_post <= end)
            ].min_since_fst_post - start
        )
    )
)

In [11]:
new_posts_times = [col for col in new_posts_times if col > 10]

In [13]:
new_posts_times

[np.float64(11.0),
 np.float64(12.0),
 np.float64(14.0),
 np.float64(18.0),
 np.float64(19.0),
 np.float64(22.0),
 np.float64(24.0),
 np.float64(26.0),
 np.float64(27.0),
 np.float64(28.0),
 np.float64(32.0),
 np.float64(35.0),
 np.float64(36.0),
 np.float64(37.0),
 np.float64(40.0),
 np.float64(45.0),
 np.float64(48.0),
 np.float64(52.0),
 np.float64(55.0),
 np.float64(57.0),
 np.float64(60.0),
 np.float64(62.0),
 np.float64(63.0),
 np.float64(68.0),
 np.float64(70.0),
 np.float64(71.0),
 np.float64(72.0),
 np.float64(73.0),
 np.float64(74.0),
 np.float64(75.0),
 np.float64(76.0),
 np.float64(78.0),
 np.float64(82.0),
 np.float64(83.0),
 np.float64(86.0),
 np.float64(87.0),
 np.float64(89.0),
 np.float64(90.0),
 np.float64(91.0),
 np.float64(92.0),
 np.float64(94.0),
 np.float64(95.0),
 np.float64(96.0),
 np.float64(99.0),
 np.float64(101.0),
 np.float64(102.0),
 np.float64(104.0),
 np.float64(106.0),
 np.float64(107.0),
 np.float64(108.0),
 np.float64(113.0),
 np.float64(114.0),
 np.

In [12]:
import mlflow
import mlflow.lightgbm
import warnings
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score, f1_score
import lightgbm as lgb


previous_node_count = 0


for time_cut in new_posts_times:
    print(f"\n=== Time Cut: {time_cut} minutes ===")
    
    processor = Load_Rumours_Dataset_filtering_since_first_post(file_path_replies, file_path_posts, time_cut=time_cut)
    processor.load_data()
    processor.process_data()
    train, test = processor.get_final_dataframes()

    # Prepare features and labels
    X_train  = train.drop(columns=['rumour'])
    X_train = np.hstack([X_train.drop(columns=['embeddings_avg']).values, np.array(pd.DataFrame(X_train.embeddings_avg.tolist()))])
    #X = np.hstack([X.drop(columns=['embeddings_avg']).values, np.array(pd.DataFrame(X.embeddings_avg.tolist()))])
    y_train =train['rumour']
    
    X_test  = test.drop(columns=['rumour'])
    X_test_new = test.iloc[previous_node_count:].drop(columns=['rumour'])
    
    X_test_new =  np.hstack([X_test_new.drop(columns=['embeddings_avg']).values, np.array(pd.DataFrame(X_test_new.embeddings_avg.tolist()))])
    X_test = np.hstack([X_test.drop(columns=['embeddings_avg']).values, np.array(pd.DataFrame(X_test.embeddings_avg.tolist()))])
    
    
    y_test =test['rumour']
    y_test_new = test.iloc[previous_node_count:]['rumour']
    
    previous_node_count = test.shape[0]
    
    print(f"New Instances: {X_test_new.shape[0]}")


    model = lgb.LGBMClassifier(
           
           **best_params,
       
    )

    with mlflow.start_run(run_name=f"time_cut_{time_cut}"):
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            model.fit(
                X_train, y_train,
                eval_metric=["binary_logloss", "auc"]
            )

        # Get predicted probabilities
        y_train_prob = model.predict_proba(X_train)[:, 1]
        y_test_prob = model.predict_proba(X_test)[:, 1]
        current_df_metrics = df_metrics.iloc[X_train.shape[0]:X_train.shape[0]+X_test.shape[0]]
        current_df_metrics['prob'] = y_test_prob
        df_metrics_by_bucket = compute_metrics_custom(current_df_metrics)
        
        if X_test_new.shape[0] >0:
             y_test_new_prob = model.predict_proba(X_test_new)[:, 1]
             new_df_metrics = current_df_metrics.iloc[-X_test_new.shape[0]:]
             new_df_metrics['prob'] = y_test_new_prob
             new_df_metrics_by_bucket = compute_metrics_custom(new_df_metrics)
        else:
            continue

        # Find best threshold maximizing F1 score on training data
        thresholds = np.linspace(0.01, 0.99, 100)
        f1_scores = [f1_score(y_train, (y_train_prob > t).astype(int)) for t in thresholds]
        best_idx = np.argmax(f1_scores)
        best_threshold = thresholds[best_idx]

        # Apply optimal threshold
        y_train_pred = (y_train_prob > best_threshold).astype(int)
        y_test_pred = (y_test_prob > best_threshold).astype(int)
        y_test_new_pred = (y_test_new_prob > best_threshold).astype(int)

        # Log train metrics
        mlflow.log_metric("train_accuracy", accuracy_score(y_train, y_train_pred))
        mlflow.log_metric("train_precision", precision_score(y_train, y_train_pred))
        mlflow.log_metric("train_recall", recall_score(y_train, y_train_pred))
        mlflow.log_metric("train_f1", f1_score(y_train, y_train_pred))
        mlflow.log_metric("train_auc", roc_auc_score(y_train, y_train_prob))

        # Log test metrics
        mlflow.log_metric("final_acc", accuracy_score(y_test, y_test_pred))
        mlflow.log_metric("final_precision", precision_score(y_test, y_test_pred))
        mlflow.log_metric("final_recall", recall_score(y_test, y_test_pred))
        mlflow.log_metric("final_f1", f1_score(y_test, y_test_pred))
        mlflow.log_metric("final_auc", roc_auc_score(y_test, y_test_prob))
        mlflow.log_metric("new_posts", X_test_new.shape[0])
        new_df_metrics_by_bucket.to_csv(f"metrics_by_bucket_new_posts_{event_name}_LGBM.csv", index=False)
        mlflow.log_artifact(f"metrics_by_bucket_new_posts_{event_name}_LGBM.csv")

        if X_test_new.shape[0] >0:
            mlflow.log_metric("curr_precision", precision_score(y_test_new, y_test_new_pred))
            mlflow.log_metric("curr_recall", recall_score(y_test_new, y_test_new_pred))
            mlflow.log_metric("curr_acc", accuracy_score(y_test_new, y_test_new_pred))
        else:
            mlflow.log_metric("curr_precision", 0)
            mlflow.log_metric("curr_recall",0)
            mlflow.log_metric("curr_acc", 0)
            

        # Log threshold and time_cut
        mlflow.log_metric("optimal_threshold", best_threshold)
        mlflow.log_metric("time_cut", time_cut)



=== Time Cut: 11.0 minutes ===
New Instances: 8

=== Time Cut: 12.0 minutes ===
New Instances: 2

=== Time Cut: 14.0 minutes ===
New Instances: 1

=== Time Cut: 18.0 minutes ===
New Instances: 2

=== Time Cut: 19.0 minutes ===
New Instances: 1

=== Time Cut: 22.0 minutes ===
New Instances: 2

=== Time Cut: 24.0 minutes ===
New Instances: 1

=== Time Cut: 26.0 minutes ===
New Instances: 3

=== Time Cut: 27.0 minutes ===
New Instances: 0

=== Time Cut: 28.0 minutes ===
New Instances: 1

=== Time Cut: 32.0 minutes ===
New Instances: 3

=== Time Cut: 35.0 minutes ===
New Instances: 2

=== Time Cut: 36.0 minutes ===
New Instances: 0

=== Time Cut: 37.0 minutes ===
New Instances: 1

=== Time Cut: 40.0 minutes ===
New Instances: 3

=== Time Cut: 45.0 minutes ===
New Instances: 2

=== Time Cut: 48.0 minutes ===
New Instances: 3

=== Time Cut: 52.0 minutes ===
New Instances: 1

=== Time Cut: 55.0 minutes ===
New Instances: 3

=== Time Cut: 57.0 minutes ===
New Instances: 0

=== Time Cut: 60.0 